# Chapter 4: From Gradient Boosting to XGBoost

# 1. Load Libraries

In [4]:
# Import pandas and numpy
import pandas as pd
import numpy as np

# Silence warnings
import warnings
warnings.filterwarnings('ignore')

# 2. Load dataset

In [5]:
df_bikes = pd.read_csv('bike_rentals_cleaned.csv')
df_bikes.head()

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,1.0,0.0,1.0,0.0,6.0,0.0,2,0.344167,0.363625,0.805833,0.160446,985
1,2,1.0,0.0,1.0,0.0,0.0,0.0,2,0.363478,0.353739,0.696087,0.248539,801
2,3,1.0,0.0,1.0,0.0,1.0,1.0,1,0.196364,0.189405,0.437273,0.248309,1349
3,4,1.0,0.0,1.0,0.0,2.0,1.0,1,0.200000,0.212122,0.590435,0.160296,1562
4,5,1.0,0.0,1.0,0.0,3.0,1.0,1,0.226957,0.229270,0.436957,0.186900,1600


# 3. Split the dataset

## 3.1 Split the dataset into input features (X) and target label (y)

In [7]:
# X_bikes will contain all columns except the last one.
# iloc[:, :-1] selects all rows (:) and all columns except the last one (:-1).
# These are usually the input features used to make predictions.
X_bikes = df_bikes.iloc[:, :-1]

# y_bikes will contain only the last column.
# iloc[:, -1] selects all rows and only the last column (-1).
# This is typically the target variable we are trying to predict.
y_bikes = df_bikes.iloc[:, -1]

X_bikes.shape, y_bikes.shape

((731, 12), (731,))

## 3.2 Split the dataset into training and test sets

In [8]:
# Import the train_test_split function from scikit-learn
# This function helps us split the data into training and testing sets
from sklearn.model_selection import train_test_split

# Split the data into training and testing subsets
# X_train and y_train will be used to train the model
# X_test and y_test will be used to test how well the model performs on unseen data
# random_state=2 ensures the split is reproducible (you get the same result every time you run the code)
X_train, X_test, y_train, y_test = train_test_split(X_bikes, y_bikes, random_state=2)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((548, 12), (183, 12), (548,), (183,))

# 4.0 Building a gradient boosting model from scratch

## 4.1 Fit the data to the decision tree

You may use a decision **tree stump**, which has a **max_depth value of 1**, or a decision tree with a **max_depth value of 2 or 3**. The initial decision tree, called a **base learner**, *should not be fine-tuned for accuracy*.

We want a model that focuses on learning from errors, not a model that relies heavily on the **base learner**.

Initialize a decision tree with **max_depth=2** and fit it on the training set as tree_1, since it's the **first tree** in our ensemble

In [9]:
# Import the DecisionTreeRegressor class from scikit-learn
# This is a type of model used for regression tasks (predicting continuous values)
from sklearn.tree import DecisionTreeRegressor

# Create (initialize) an instance of the Decision Tree Regressor
# max_depth=2 means the tree can split only twice, limiting its complexity
# random_state=2 makes the result reproducible (so the same splits happen every time)
tree_1 = DecisionTreeRegressor(max_depth=2, random_state=2)

# Train (fit) the decision tree model using the training data
# The model learns patterns from X_train (features) to predict y_train (target)
tree_1.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=2, random_state=2)

## 4.2 Make predictions with the training set (and not test set)

Instead of making predictions with the test set, predictions in gradient boosting are initially made with the training set.

**Why?**

To compute the residuals, we need to compare the predictions while still in the training phase.

The test phase of the model build comes at the end, after all the trees have been constructed.

The predictions of the training set for the first round are obtained by adding the predict method to tree_1 with X_train as the input:

In [10]:
# Use the trained decision tree model to make predictions on the training data

# tree_1.predict(X_train) applies the learned rules from the training process to the training inputs
# It outputs the predicted values for each row in the training set

# These predictions are stored in y_train_pred
# We can later compare them to the actual values (y_train) to evaluate model performance
y_train_pred = tree_1.predict(X_train)

In [11]:
# Compute the residuals (errors) from the first tree's predictions

# y_train is the actual target value
# y_train_pred is the predicted value from tree_1
# The residual is the difference between the actual and predicted values
# This tells us how much tree_1 got wrong for each training example
y2_train = y_train - y_train_pred

# Initialize a second Decision Tree Regressor
# This new tree will try to learn the pattern in the residuals (the errors)
# max_depth=2 again limits the complexity, and random_state ensures reproducibility
tree_2 = DecisionTreeRegressor(max_depth=2, random_state=2)

# Fit the second tree to the same input features (X_train), but now the targets are the residuals (y2_train)
# The goal is for this tree to learn how to correct the mistakes made by the first tree
tree_2.fit(X_train, y2_train)

DecisionTreeRegressor(max_depth=2, random_state=2)

In [ ]:
# Make predictions on training set
y2_train_pred = tree_2.predict(X_train)

# Compute residuals
y3_train = y2_train - y2_train_pred

# Initialize Decision Tree Regressor
tree_3 = DecisionTreeRegressor(max_depth=2, random_state=2)

# Fit tree to training data
tree_3.fit(X_train, y3_train)

DecisionTreeRegressor(max_depth=2, random_state=2)

In [ ]:
y1_pred = tree_1.predict(X_test)

y2_pred = tree_2.predict(X_test)

y3_pred = tree_3.predict(X_test)

y_pred = y1_pred + y2_pred + y3_pred

# Import mean_squared_error
from sklearn.metrics import mean_squared_error as MSE

# Compute root mean squared error (rmse)
MSE(y_test, y_pred)**0.5


911.0479538776444

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
gbr = GradientBoostingRegressor(max_depth=2, n_estimators=3, random_state=2, learning_rate=1.0)

gbr.fit(X_train, y_train)

# Predict test data
y_pred = gbr.predict(X_test)

# Compute root mean squared error (rmse)
MSE(y_test, y_pred)**0.5

911.0479538776439

In [ ]:
gbr = GradientBoostingRegressor(max_depth=2, n_estimators=30, random_state=2, learning_rate=1.0)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)
MSE(y_test, y_pred)**0.5

857.1072323426944

In [ ]:
gbr = GradientBoostingRegressor(max_depth=2, n_estimators=300, random_state=2, learning_rate=1.0)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)
MSE(y_test, y_pred)**0.5

936.3617413678853

In [ ]:
gbr = GradientBoostingRegressor(max_depth=2, n_estimators=300, random_state=2)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)
MSE(y_test, y_pred)**0.5

653.7456840231495

In [ ]:
learning_rate_values = [0.001, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]
for value in learning_rate_values:
    gbr = GradientBoostingRegressor(max_depth=2, n_estimators=300, random_state=2, learning_rate=value)
    gbr.fit(X_train, y_train)
    y_pred = gbr.predict(X_test)
    rmse = MSE(y_test, y_pred)**0.5
    print('Learning Rate:', value, ', Score:', rmse)

Learning Rate: 0.001 , Score: 1633.0261400367258
Learning Rate: 0.01 , Score: 831.5430182728547
Learning Rate: 0.05 , Score: 685.0192988749717
Learning Rate: 0.1 , Score: 653.7456840231495
Learning Rate: 0.15 , Score: 687.666134269379
Learning Rate: 0.2 , Score: 664.312804425697
Learning Rate: 0.3 , Score: 689.4190385930236
Learning Rate: 0.5 , Score: 693.8856905068778
Learning Rate: 1.0 , Score: 936.3617413678853


In [ ]:
depths = [None, 1, 2, 3, 4]
for depth in depths:
    gbr = GradientBoostingRegressor(max_depth=depth, n_estimators=300, random_state=2)
    gbr.fit(X_train, y_train)
    y_pred = gbr.predict(X_test)
    rmse = MSE(y_test, y_pred)**0.5
    print('Max Depth:', depth, ', Score:', rmse)

Max Depth: None , Score: 869.2783041945797
Max Depth: 1 , Score: 707.8261886858736
Max Depth: 2 , Score: 653.7456840231495
Max Depth: 3 , Score: 646.4045923317708
Max Depth: 4 , Score: 663.048387855927


In [ ]:
samples = [1, 0.9, 0.8, 0.7, 0.6, 0.5]
for sample in samples:
    gbr = GradientBoostingRegressor(max_depth=3, n_estimators=300, subsample=sample, random_state=2)
    gbr.fit(X_train, y_train)
    y_pred = gbr.predict(X_test)
    rmse = MSE(y_test, y_pred)**0.5
    print('Subsample:', sample, ', Score:', rmse)

Subsample: 1 , Score: 646.4045923317708
Subsample: 0.9 , Score: 620.1819001443569
Subsample: 0.8 , Score: 617.2355650565677
Subsample: 0.7 , Score: 612.9879156983139
Subsample: 0.6 , Score: 622.6385116402317
Subsample: 0.5 , Score: 626.9974073227554


In [ ]:
params={'subsample':[0.65, 0.7, 0.75],
                          'n_estimators':[300, 500, 1000],
                          'learning_rate':[0.05, 0.075, 0.1]
                         }

# Import RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV

gbr = GradientBoostingRegressor(max_depth=3, random_state=2)


# Instantiate RandomizedSearchCV as rand_reg
rand_reg = RandomizedSearchCV(gbr, params, n_iter=10, scoring='neg_mean_squared_error',
                              cv=5, n_jobs=-1, random_state=2)

# Fit grid_reg on X_train and y_train
rand_reg.fit(X_train, y_train)

# Extract best estimator
best_model = rand_reg.best_estimator_

# Extract best params
best_params = rand_reg.best_params_

# Print best params
print("Best params:", best_params)

# Compute best score
best_score = np.sqrt(-rand_reg.best_score_)

# Print best score
print("Training score: {:.3f}".format(best_score))

# Predict test set labels
y_pred = best_model.predict(X_test)

# Compute rmse_test
rmse_test = MSE(y_test, y_pred)**0.5

# Print rmse_test
print('Test set score: {:.3f}'.format(rmse_test))

Best params: {'subsample': 0.65, 'n_estimators': 300, 'learning_rate': 0.05}
Training score: 636.200
Test set score: 625.985


In [ ]:
gbr = GradientBoostingRegressor(max_depth=3, n_estimators=1600, subsample=0.75, learning_rate=0.02, random_state=2)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)
MSE(y_test, y_pred)**0.5

596.9544588974487

In [ ]:
# Import XGBRegressor
from xgboost import XGBRegressor

# Instantiate the XGBRegressor, xg_reg
xg_reg = XGBRegressor(max_depth=3, n_estimators=1600, eta=0.02, subsample=0.75, random_state=2)

# Fit xg_reg to training set
xg_reg.fit(X_train, y_train)

# Predict labels of test set, y_pred
y_pred = xg_reg.predict(X_test)

# Compute root mean squared error (rmse)
MSE(y_test, y_pred)**0.5

584.339544309016

In [ ]:
df = pd.read_csv('exoplanets.csv')
df.head()

,LABEL,FLUX.1,FLUX.2,FLUX.3,FLUX.4,FLUX.5,FLUX.6,FLUX.7,FLUX.8,FLUX.9,...,FLUX.3188,FLUX.3189,FLUX.3190,FLUX.3191,FLUX.3192,FLUX.3193,FLUX.3194,FLUX.3195,FLUX.3196,FLUX.3197
0,2,93.85,83.81,20.10,-26.98,-39.56,-124.71,-135.18,-96.27,-79.89,...,-78.07,-102.15,-102.15,25.13,48.57,92.54,39.32,61.42,5.08,-39.54
1,2,-38.88,-33.83,-58.54,-40.09,-79.31,-72.81,-86.55,-85.33,-83.97,...,-3.28,-32.21,-32.21,-24.89,-4.86,0.76,-11.70,6.46,16.00,19.93
2,2,532.64,535.92,513.73,496.92,456.45,466.00,464.50,486.39,436.56,...,-71.69,13.31,13.31,-29.89,-20.88,5.06,-11.80,-28.91,-70.02,-96.67
3,2,326.52,347.39,302.35,298.13,317.74,312.70,322.33,311.31,312.42,...,5.71,-3.73,-3.73,30.05,20.03,-12.67,-8.77,-17.31,-17.35,13.98
4,2,-1107.21,-1112.59,-1118.95,-1095.10,-1057.55,-1034.48,-998.34,-1022.71,-989.57,...,-594.37,-401.66,-401.66,-357.24,-443.76,-438.54,-399.71,-384.65,-411.79,-510.54


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5087 entries, 0 to 5086
Columns: 3198 entries, LABEL to FLUX.3197
dtypes: float64(3197), int64(1)
memory usage: 124.1 MB


In [ ]:
df.isnull().sum().sum()

0

In [ ]:
# Split data into X and y
X = df.iloc[:,1:]
y = df.iloc[:,0]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=2)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
# Import XGBRegressor
from xgboost import XGBClassifier

# Import accuracy_score
from sklearn.metrics import accuracy_score

In [ ]:
import time
start = time.time()

df.info()

end = time.time()
elapsed = end - start

print('\nRun Time: ' + str(elapsed) + ' seconds.')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5087 entries, 0 to 5086
Columns: 3198 entries, LABEL to FLUX.3197
dtypes: float64(3197), int64(1)
memory usage: 124.1 MB

Run Time: 0.02074909210205078 seconds.


In [ ]:
start = time.time()

gbr = GradientBoostingClassifier(n_estimators=100, max_depth=2, random_state=2)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)
score = accuracy_score(y_pred, y_test)
print('Score: ' + str(score))

end = time.time()
elapsed = end - start

print('Run Time: ' + str(elapsed) + ' seconds')

Score: 0.9874213836477987
Run Time: 205.1609354019165 seconds


In [ ]:
start = time.time()

# Instantiate the XGBRegressor, xg_reg
xg_reg = XGBClassifier(n_estimators=100, max_depth=2, random_state=2)

# Fit xg_reg to training set
xg_reg.fit(X_train, y_train)

# Predict labels of test set, y_pred
y_pred = xg_reg.predict(X_test)

score = accuracy_score(y_pred, y_test)

print('Score: ' + str(score))

end = time.time()
elapsed = end - start

print('Run Time: ' + str(elapsed) + ' seconds')

Score: 0.9913522012578616
Run Time: 9.091089010238647 seconds
